In [1]:
!pip install rectools[torch] lightning-fabric

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
%cd "/content/drive/MyDrive/RecSysCourse/hw2"

/content/drive/MyDrive/RecSysCourse/hw2


In [4]:
import os
import glob
import json

import pandas as pd
import numpy as np

from rectools import Columns
from rectools.dataset import Dataset
from rectools.models import PopularModel, BERT4RecModel
from rectools.metrics import NDCG, HitRate
from rectools.visuals.visual_app import ItemToItemVisualApp

from lightning_fabric import seed_everything

In [5]:
# Enable deterministic behaviour with CUDA >= 10.2
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
RANDOM_STATE = 42
seed_everything(RANDOM_STATE, workers=True)

DATA_DIR = "./data/"

INFO:lightning_fabric.utilities.seed:Seed set to 42


# Prepare dataset

In [6]:
train_data = pd.concat([
    pd.read_json(data_path, lines=True)[["user", "track", "timestamp", "time"]]
    for data_path
    in glob.glob(DATA_DIR + "train/*/data.json")
])

# Use rectools column names
train_interactions = train_data[
    train_data["time"] > 0.6
].copy().rename(columns={
    "user": Columns.User,
    "track": Columns.Item,
    "timestamp": Columns.Datetime,
    "time": Columns.Weight
})

print(f"Train data size: {train_data.shape[0]}, train interactions: {train_interactions.shape[0]}")
train_interactions.head(3)

Train data size: 1034924, train interactions: 446804


,user_id,item_id,datetime,weight
0,5984,2748,2026-04-18 10:40:51.461,1.00
1,5984,8807,2026-04-18 10:40:51.482,0.81
2,5984,9153,2026-04-18 10:40:51.495,0.67


In [7]:
test_data = pd.concat([
    pd.read_json(data_path, lines=True)[["user", "track", "timestamp", "time"]]
    for data_path
    in glob.glob(DATA_DIR + "test/*/data.json")
])

# Use rectools column names
test_interactions = test_data[
    (test_data["time"] > 0.6)
    & (test_data["user"].isin(train_interactions[Columns.User]))
    & (test_data["track"].isin(train_interactions[Columns.Item]))
].copy().rename(columns={
    "user": Columns.User,
    "track": Columns.Item,
    "timestamp": Columns.Datetime,
    "time": Columns.Weight
})

print(f"Test data size: {test_data.shape[0]}, test interactions: {test_interactions.shape[0]}")
test_interactions.head(3)

Test data size: 104877, test interactions: 46469


,user_id,item_id,datetime,weight
0,2426,3,2026-04-18 11:19:52.666,1.00
1,2426,1,2026-04-18 11:19:52.674,0.80
2,2426,9,2026-04-18 11:19:52.677,0.64


In [8]:
tracks = pd.read_json(DATA_DIR + "tracks.json", lines=True).drop_duplicates(subset=["track"]).rename(columns={"track": Columns.Item})

items = tracks.loc[tracks[Columns.Item].isin(train_interactions[Columns.Item])].copy()

artist_feature = items[[Columns.Item, "artist_id"]].copy()
artist_feature.columns = ["id", "value"]
artist_feature["feature"] = "artist"

genre_feature = items[[Columns.Item, "genres"]].explode("genres").copy()
genre_feature.columns = ["id", "value"]
genre_feature["feature"] = "genre"

mood_feature = items[[Columns.Item, "mood"]].copy()
mood_feature.columns = ["id", "value"]
mood_feature["feature"] = "mood"

country_feature = items[[Columns.Item, "artist_country"]].copy()
country_feature.columns = ["id", "value"]
country_feature["feature"] = "country"

year_feature = items[[Columns.Item, "year"]].copy()
year_feature.columns = ["id", "value"]
year_feature["feature"] = "year"

item_features = pd.concat((artist_feature, genre_feature, mood_feature, country_feature, year_feature))

print(f"Total tracks: {tracks.shape[0]}, selected tracks: {items.shape[0]}")
item_features.head(3)

Total tracks: 16198, selected tracks: 16097


,id,value,feature
0,0,0,artist
1,1,0,artist
2,2,0,artist


In [9]:
train_dataset = Dataset.construct(
    interactions_df=train_interactions,
    item_features_df=item_features,
    cat_item_features=["artist", "genre", "mood", "country", "year"],
)

test_dataset = Dataset.construct(
    interactions_df=test_interactions,
    item_features_df=item_features,
    cat_item_features=["artist", "genre", "mood", "country", "year"],
)

# Train model

In [10]:
# Popular model (for comparison)
popular = PopularModel()
popular.fit(train_dataset)
popular_recs = popular.recommend(
    users=test_interactions[Columns.User].unique(),
    dataset=train_dataset,
    k=10,
    filter_viewed=True,
)

In [11]:
# BERT4Rec model
bert4rec = BERT4RecModel(
    session_max_len=100,      # Maximum length of user sequence
    mask_prob=0.15,           # Probability of masking an item in interactions sequence
    loss="softmax",           # Loss function
    n_factors=64,             # Latent embeddings size
    n_blocks=1,               # Number of transformer blocks
    n_heads=4,                # Number of attention heads
    dropout_rate=0.2,         # Probability of a hidden unit to be zeroed
    lr=0.001,                 # Learning rate
    batch_size=128,           # Batch size
    epochs=100,               # Exact number of training epochs
    verbose=1,                # Verbosity level
    deterministic=True        # For reproducibility of results
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [12]:
%%time
bert4rec.fit(train_dataset)

/usr/local/lib/python3.12/dist-packages/rectools/dataset/identifiers.py:60: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  unq_values = pd.unique(values)
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='item_net_block_types', input_value=('rectools.models.nn.item...net.CatFeaturesItemNet'), input_type=tuple])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name        | Type                     | Params | Mode 
--

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


CPU times: user 19min 22s, sys: 2.8 s, total: 19min 25s
Wall time: 19min 38s


In [13]:
bert4rec_recs = bert4rec.recommend(
    users=test_interactions[Columns.User].unique(),
    dataset=train_dataset,
    k=10,
    filter_viewed=True,
    on_unsupported_targets="warn"
)

In [14]:
metrics = {
  "ndcg": NDCG(k=10),
  "hit_rate": HitRate(k=10),
}

recs = {
    "popular": popular_recs,
    "bert4rec": bert4rec_recs,
}

results = []
for metric_name, metric in metrics.items():
  for model, model_recs in recs.items():
    results.append({
        "model": model,
        "metric": metric_name,
        "value": metric.calc(model_recs, interactions=test_interactions),
    })

pd.DataFrame(results).pivot(index="model", columns=["metric"], values="value").sort_values("ndcg")

metric,hit_rate,ndcg
model,,
popular,0.021844,0.006905
bert4rec,0.162718,0.035690


# Recommendations
Get I2I recommendations (10 per item)

In [15]:
item_ids = train_interactions.groupby(Columns.Item).size().sort_values(ascending=False).index.to_list() #[:15000]

bert4rec_10_i2i = bert4rec.recommend_to_items(
    target_items=item_ids,
    dataset=train_dataset,
    k=10,
    filter_itself=True,
    items_to_recommend=None,
    on_unsupported_targets="warn"
)

/usr/local/lib/python3.12/dist-packages/rectools/models/base.py:728: UserWarning: 
                Model `<class 'rectools.models.nn.transformers.bert4rec.BERT4RecModel'>` doesn't support recommendations for cold items,
                but some of given items are cold: they are not in the `dataset.item_id_map`
            
  warnings.warn(explanation)


Let's now recommend only tracks by alternative artists (delete artist of current track from recommendations)

In [16]:
bert4rec_alt_i2i = bert4rec.recommend_to_items(
    target_items=item_ids,
    dataset=train_dataset,
    k=100,
    filter_itself=True,
    items_to_recommend=None,
    on_unsupported_targets="warn"
)

/usr/local/lib/python3.12/dist-packages/rectools/models/base.py:728: UserWarning: 
                Model `<class 'rectools.models.nn.transformers.bert4rec.BERT4RecModel'>` doesn't support recommendations for cold items,
                but some of given items are cold: they are not in the `dataset.item_id_map`
            
  warnings.warn(explanation)


In [17]:
# Add artist_id for target tracks
bert4rec_alt_i2i = bert4rec_alt_i2i.merge(
    items[[Columns.Item, "artist_id"]].rename(columns={Columns.Item: "target_item_id", "artist_id": "target_artist_id"}),
    on="target_item_id",
    how="inner"
)

# Add artist_id for recommended tracks
bert4rec_alt_i2i = bert4rec_alt_i2i.merge(
    items[[Columns.Item, "artist_id"]].rename(columns={"artist_id": "rec_artist_id"}),
    on=Columns.Item,
    how="inner"
)

# Filter
bert4rec_alt_i2i = bert4rec_alt_i2i[
    bert4rec_alt_i2i["target_artist_id"] != bert4rec_alt_i2i["rec_artist_id"]
]

# Drop temp columns, get first 10 tracks and recalc rank
bert4rec_alt_i2i = bert4rec_alt_i2i.drop(["target_artist_id", "rec_artist_id"], axis=1)
bert4rec_alt_i2i = bert4rec_alt_i2i.groupby("target_item_id").head(10)
bert4rec_alt_i2i["rank"] = bert4rec_alt_i2i.groupby("target_item_id").cumcount() + 1

In [18]:
print(bert4rec_10_i2i.shape[0])
print(bert4rec_alt_i2i.shape[0])

160640
160640


In [19]:
ItemToItemVisualApp.construct(
    reco={"bert4rec_10": bert4rec_10_i2i, "bert4rec_alt": bert4rec_alt_i2i},
    item_data=items[[Columns.Item, "title", "artist", "genres", "artist_country"]],
    n_random_items=6
)

In [20]:
def i2i_to_json(i2i_df, file_name):
    with open(file_name, "w") as json_file:
        for target_item_id, group in i2i_df.groupby('target_item_id'):
            recommendations = group.sort_values('rank')['item_id'].tolist()
            json_file.write(json.dumps({'item_id': target_item_id, 'recommendations': recommendations}) + "\n")


i2i_to_json(bert4rec_alt_i2i, DATA_DIR + "bert4rec_alt_i2i.jsonl")

In [21]:
!head ./data/bert4rec_alt_i2i.jsonl

{"item_id": 0, "recommendations": [7134, 7133, 1031, 7118, 1332, 3638, 6942, 12744, 15403, 3417]}
{"item_id": 1, "recommendations": [3417, 13, 7134, 7133, 5829, 3423, 14330, 1031, 3406, 14408]}
{"item_id": 2, "recommendations": [7133, 7134, 3417, 13, 5829, 3423, 14330, 6942, 1031, 12258]}
{"item_id": 3, "recommendations": [3423, 3417, 13, 5829, 3406, 7133, 6974, 14205, 7134, 7104]}
{"item_id": 4, "recommendations": [14408, 15986, 6942, 14474, 14407, 15990, 15913, 3404, 7114, 16176]}
{"item_id": 5, "recommendations": [6942, 14408, 15986, 14474, 14407, 15913, 3404, 7114, 3405, 15990]}
{"item_id": 6, "recommendations": [7134, 7133, 7118, 11450, 1031, 6942, 15986, 15336, 14408, 7077]}
{"item_id": 7, "recommendations": [7134, 7133, 3417, 1031, 13, 7118, 773, 6942, 5829, 14330]}
{"item_id": 8, "recommendations": [3417, 7133, 14330, 13, 7104, 7134, 3423, 12258, 5829, 15336]}
{"item_id": 9, "recommendations": [3417, 3423, 5829, 13, 7134, 7133, 7104, 3406, 14205, 14407]}
